# Pitch Vision — Phase 0 & 1 (run this in Colab)

Runs the two phases that need real compute: **Phase 0** (stock YOLO baseline — see what it gets wrong) and **Phase 1** (fine-tune YOLO on football-specific classes).

**Before running:** Runtime → Change runtime type → select **GPU** (T4 is fine, free tier).

You'll need:
1. `clip_01_0-12s.mp4` — the trimmed test clip from your `input_videos/` folder (upload it in the cell below).
2. A free Roboflow account + API key, for Part B (Settings → API Keys on roboflow.com).

In [ ]:
!pip install -q ultralytics roboflow supervision
import ultralytics
ultralytics.checks()

## Part A — Baseline: what does stock YOLO get wrong?

This is Roadmap Phase 0.1/0.2. Stock YOLOv8x only knows COCO's generic classes (`person`, `sports ball`, ...) — it doesn't know "referee" vs "player", and it's unreliable on the ball. We measure that gap here, concretely, instead of just eyeballing it.

In [ ]:
from google.colab import files
print("Upload clip_01_0-12s.mp4 (from your project's input_videos/ folder):")
uploaded = files.upload()
clip_path = list(uploaded.keys())[0]
print("Using:", clip_path)

In [ ]:
from ultralytics import YOLO

baseline_model = YOLO('yolov8x.pt')  # downloads the pretrained COCO weights automatically
baseline_results = baseline_model.predict(source=clip_path, save=True, conf=0.1)

# Peek at frame 0's raw detections — same structure the reference tutorial walks through:
# each box has cls (class id), conf (confidence), and xyxy (bounding box corners)
print(f"Frame 0: {len(baseline_results[0].boxes)} detections")
for box in baseline_results[0].boxes:
    cls_id = int(box.cls[0])
    print(baseline_model.names[cls_id], float(box.conf[0]))

In [ ]:
# Quantify the gap (Phase 0.2 deliverable) instead of just eyeballing the output video.
# COCO class 0 = 'person', class 32 = 'sports ball' — the two classes stock YOLO has that overlap with this task.

total_frames = len(baseline_results)
frames_with_ball = 0
person_counts = []

for r in baseline_results:
    classes_this_frame = [int(c) for c in r.boxes.cls]
    if 32 in classes_this_frame:
        frames_with_ball += 1
    person_counts.append(classes_this_frame.count(0))

print(f"Total frames: {total_frames}")
print(f"Frames with a ball detected: {frames_with_ball} ({100*frames_with_ball/total_frames:.1f}%)")
print(f"Avg 'person' detections per frame: {sum(person_counts)/total_frames:.1f}")
print(f"Max 'person' detections in one frame: {max(person_counts)}")
print()
print("This is your baseline gap analysis: low ball-detection %, no referee/player distinction —")
print("and if this were full-pitch broadcast footage, some of those 'person' detections would be crowd/staff, not players.")
print("Note: this drone/top-view footage has no crowd in frame, so the 'people outside the pitch' problem")
print("from the reference video mostly won't show up here — the ball-detection rate and missing ref/player split still will.")

In [ ]:
# Download the annotated baseline video to look at directly
import glob, shutil
out_dir = glob.glob('runs/detect/predict*')[-1]
shutil.make_archive('baseline_output', 'zip', out_dir)
files.download('baseline_output.zip')

## Part B — Fine-tune YOLO on football-specific classes

Roadmap Phase 1.1/1.2. Same fine-tuning workflow you already ran in AER850_Project_3 — different dataset, same idea. Trains on Roboflow's `football-players-detection` dataset (player / referee / goalkeeper / ball, ~600 images).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"  # roboflow.com -> Settings -> API Keys

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset = project.version(1).download("yolov8")
print("Downloaded to:", dataset.location)

In [ ]:
# The Ultralytics CLI trainer expects train/valid/test to sit inside a folder
# matching the dataset name — Roboflow's download doesn't nest it that way, so fix it here.
import os, shutil

nested = os.path.join(dataset.location, os.path.basename(dataset.location))
os.makedirs(nested, exist_ok=True)
for split in ['train', 'valid', 'test']:
    src = os.path.join(dataset.location, split)
    dst = os.path.join(nested, split)
    if os.path.isdir(src) and not os.path.isdir(dst):
        shutil.move(src, dst)

data_yaml = os.path.join(dataset.location, 'data.yaml')
print("data.yaml at:", data_yaml)
!cat {data_yaml}

In [ ]:
# Train. ~100 epochs on this small dataset typically takes well under an hour on a free Colab GPU.
# If Colab disconnects mid-run, re-run this cell — Ultralytics resumes from the last checkpoint
# in runs/detect/train/weights/last.pt automatically if you add resume=True.
!yolo task=detect mode=train model=yolov8x.pt data="{data_yaml}" epochs=100 imgsz=640

In [ ]:
# Grab the trained weights — this is the one file you need to bring back to your local project's models/ folder.
import glob, shutil
best_pt = glob.glob('runs/detect/train*/weights/best.pt')[-1]
shutil.copy(best_pt, 'best.pt')
files.download('best.pt')
print("Downloaded best.pt — save this into your local project's models/ folder.")

## Validate: does the fine-tuned model actually beat the baseline?

Roadmap Phase 1.2 checkpoint. Re-run on the exact same clip and compare.

In [ ]:
finetuned_model = YOLO('best.pt')
ft_results = finetuned_model.predict(source=clip_path, save=True, conf=0.1)

print("Classes this model knows:", finetuned_model.names)
print()

class_counts = {}
for r in ft_results:
    for c in r.boxes.cls:
        name = finetuned_model.names[int(c)]
        class_counts[name] = class_counts.get(name, 0) + 1

total_ft_frames = len(ft_results)
print(f"Total frames: {total_ft_frames}")
for name, count in class_counts.items():
    print(f"  {name}: {count} detections total, {count/total_ft_frames:.1f} avg/frame")

print()
print("Compare 'ball' detections/frame here against the baseline's ball-detection rate above.")
print("Also check: does it correctly separate referee from player, instead of lumping everyone as 'person'?")

out_dir = glob.glob('runs/detect/predict*')[-1]
shutil.make_archive('finetuned_output', 'zip', out_dir)
files.download('finetuned_output.zip')

### Done with Colab for now

You should have downloaded: `baseline_output.zip`, `best.pt`, `finetuned_output.zip`.

Put `best.pt` in your local project's `models/` folder — that's what Stage 2 (tracking) will load. Bring the two output videos back too if you want to compare them side by side.

**One known limitation to expect** (same as the reference video): the goalkeeper class sometimes flip-flops with 'player' on this small (~600 image) dataset. The plan is to normalize goalkeeper→player at the tracking stage in Phase 2.2, since we're not scoring goalkeepers separately — nothing to fix here.